# Khmer Word Segmentation

This notebook segments a Khmer sentence that you type, using words in a dictionary CSV. It uses a Trie and longest-match lookup; it does not train a machine-learning model.


## 1. Set the file and column names

The dictionary is in the `data` folder. This CSV uses lowercase `word` and `pos` column names, so the settings below match it.

In [22]:
DICTIONARY_CSV = "data/khmer_dictionary.csv"
WORD_COLUMN = "word"
POS_COLUMN = "pos"  # Set to None if the dictionary has no part-of-speech column.

## 2. Make a word lookup tree

A Trie stores words by their letters. `longest_match` finds the longest dictionary word beginning at a given position.

In [23]:
class TrieNode:
    def __init__(self):
        self.children = {}
        self.pos = None


class Trie:
    def __init__(self):
        self.root = TrieNode()

    def insert(self, word, pos=None):
        node = self.root
        for character in word:
            if character not in node.children:
                node.children[character] = TrieNode()
            node = node.children[character]
        node.pos = pos or "N/A"

    def longest_match(self, text, start):
        node = self.root
        best_word = None
        best_pos = None
        index = start

        while index < len(text) and text[index] in node.children:
            node = node.children[text[index]]
            index += 1
            if node.pos is not None:
                best_word = text[start:index]
                best_pos = node.pos

        if best_word is None:
            return None, None
        return best_word, best_pos

## 3. Load the dictionary CSV

This reads the dictionary into the Trie. The small example later in the notebook can also be tried on its own.

In [24]:
import csv
import unicodedata
from pathlib import Path


def load_dictionary(csv_path, word_column=WORD_COLUMN, pos_column=POS_COLUMN):
    path = Path(csv_path)
    if not path.is_file():
        raise FileNotFoundError(
            f"Dictionary not found: {path.resolve()}. "
            "Put the file there or update DICTIONARY_CSV."
        )

    trie = Trie()
    with path.open("r", encoding="utf-8-sig", newline="") as csv_file:
        reader = csv.DictReader(csv_file)
        if not reader.fieldnames or word_column not in reader.fieldnames:
            headers = reader.fieldnames or []
            raise ValueError(
                f"Word column {word_column!r} was not found. CSV headers: {headers}. "
                "Update WORD_COLUMN to match your file."
            )

        for row in reader:
            word = (row.get(word_column) or "").strip()
            if not word:
                continue
            pos = (row.get(pos_column) or "N/A").strip() if pos_column else "N/A"
            trie.insert(word, pos)

    return trie

## 4. Segment text

The segmenter scans left to right. At each Khmer character it chooses the longest matching dictionary word. If it cannot find one, it keeps that character (and following Khmer combining marks) in the output and records it as unmatched. Whitespace is preserved; non-Khmer text and punctuation are kept rather than discarded.

In [25]:
def is_khmer_letter(character):
    codepoint = ord(character)
    return 0x1780 <= codepoint <= 0x17D3


def next_unknown_unit(text, start):
    """Keep a Khmer character together with any following combining marks."""
    end = start + 1
    while end < len(text) and unicodedata.category(text[end]).startswith("M"):
        end += 1
    return text[start:end]


def segment_chunk(text, trie):
    tokens = []
    labels = []
    unmatched = []
    index = 0

    while index < len(text):
        if is_khmer_letter(text[index]):
            word, pos = trie.longest_match(text, index)
            if word is not None:
                tokens.append(word)
                labels.append((word, pos))
                index += len(word)
            else:
                unknown_start = index
                index += len(next_unknown_unit(text, index))
                while index < len(text) and is_khmer_letter(text[index]):
                    next_word, _ = trie.longest_match(text, index)
                    if next_word is not None:
                        break
                    index += len(next_unknown_unit(text, index))
                unknown = text[unknown_start:index]
                tokens.append(unknown)
                labels.append((unknown, "UNKNOWN"))
                unmatched.append(unknown)
        else:
            end = index + 1
            while end < len(text) and not is_khmer_letter(text[end]):
                end += 1
            other_text = text[index:end]
            tokens.append(other_text)
            labels.append((other_text, "N/A"))
            index = end

    return tokens, labels, unmatched


def segment_text(text, trie):
    """Segment one string, preserving whitespace and returning labels and unknowns."""
    output_parts = []
    all_labels = []
    all_unmatched = []
    index = 0

    while index < len(text):
        if text[index].isspace():
            end = index + 1
            while end < len(text) and text[end].isspace():
                end += 1
            output_parts.append(text[index:end])
            index = end
        else:
            end = index + 1
            while end < len(text) and not text[end].isspace():
                end += 1
            tokens, labels, unmatched = segment_chunk(text[index:end], trie)
            output_parts.append(" ".join(tokens))
            all_labels.extend(labels)
            all_unmatched.extend(unmatched)
            index = end

    return "".join(output_parts), all_labels, all_unmatched

## 5. Try a tiny example first

These sample words are only for demonstrating the code. Replace them by loading your teacher's dictionary when it is ready.

In [26]:
demo_trie = Trie()
demo_trie.insert("សួស្តី", "interjection")
demo_trie.insert("ខ្មែរ", "noun")

demo_text = "សួស្តីខ្មែរ"
demo_result, demo_labels, demo_unmatched = segment_text(demo_text, demo_trie)

print("Input:     ", demo_text)
print("Segmented:", demo_result)
print("Words/POS:", demo_labels)
print("Unmatched:", demo_unmatched)

Input:      សួស្តីខ្មែរ
Segmented: សួស្តី ខ្មែរ
Words/POS: [('សួស្តី', 'interjection'), ('ខ្មែរ', 'noun')]
Unmatched: []


## 6. Type your sentence

Replace the sample below with your Khmer sentence. This is the only input method for now.

### Option A: Type or paste text

In [31]:
typed_text = "រួមមានសំណួរប្រវត្តិវិទ្យា សំណួរចាស់ៗ និងសំណួរបើក សម្រាប់ទាំងថ្នាក់វិទ្យាសាស្ត្រពិត និងវិទ្យាសាស្ត្រសង្គម"  # Replace this with your own Khmer sentence.
texts_to_segment = [typed_text]

## 7. Segment your sentence

Run this cell after typing your sentence. It shows the segmented sentence, the same words separated by zero-width spaces, total word count, repeated words with their counts, and any unmatched Khmer text.

In [32]:
from collections import Counter

dictionary_trie = load_dictionary(DICTIONARY_CSV)

for number, text in enumerate(texts_to_segment, start=1):
    segmented, word_labels, unmatched = segment_text(text, dictionary_trie)
    print(f"Text {number}:")
    print("Original:  ", text)
    print("Segmented:", segmented)
    print("Words/POS:", word_labels)
    zero_space = "\u200b".join(word.strip() for word, _ in word_labels if word.strip())
    print("ZeroSpace:", zero_space)
    word_tokens = [
        word for word, _ in word_labels
        if any(is_khmer_letter(char) or ord(char) == 0x17D7 for char in word)
    ]
    print("Total Words:", len(word_tokens))
    repeated_words = {word: count for word, count in Counter(word_tokens).items() if count > 1}
    print("Repeated Words:", repeated_words if repeated_words else "None")
    print("Unmatched:", unmatched)
    print()

Text 1:
Original:   រួមមានសំណួរប្រវត្តិវិទ្យា សំណួរចាស់ៗ និងសំណួរបើក សម្រាប់ទាំងថ្នាក់វិទ្យាសាស្ត្រពិត និងវិទ្យាសាស្ត្រសង្គម
Segmented: រួមមាន សំណួរ ប្រវត្តិវិទ្យា សំណួរ ចាស់ ៗ និង សំណួរ បើក សម្រាប់ ទាំង ថ្នាក់ វិទ្យាសាស្ត្រ ពិត និង វិទ្យាសាស្ត្រ សង្គម
Words/POS: [('រួមមាន', 'N/A'), ('សំណួរ', 'ន.'), ('ប្រវត្តិវិទ្យា', 'ន.'), ('សំណួរ', 'ន.'), ('ចាស់', 'គុ.'), ('ៗ', 'N/A'), ('និង', 'ឈ.'), ('សំណួរ', 'ន.'), ('បើក', 'កិ.'), ('សម្រាប់', 'គុ.'), ('ទាំង', 'ឈ.'), ('ថ្នាក់', 'ន.'), ('វិទ្យាសាស្ត្រ', 'ន.'), ('ពិត', 'គុ.'), ('និង', 'ឈ.'), ('វិទ្យាសាស្ត្រ', 'ន.'), ('សង្គម', 'ន.')]
ZeroSpace: រួមមាន​សំណួរ​ប្រវត្តិវិទ្យា​សំណួរ​ចាស់​ៗ​និង​សំណួរ​បើក​សម្រាប់​ទាំង​ថ្នាក់​វិទ្យាសាស្ត្រ​ពិត​និង​វិទ្យាសាស្ត្រ​សង្គម
Total Words: 17
Repeated Words: {'សំណួរ': 3, 'និង': 2, 'វិទ្យាសាស្ត្រ': 2}
Unmatched: []

